## 准备数据

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [3]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [4]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([28 * 28, 128], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([128]))
        self.W2 = tf.Variable(tf.random.normal([128, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 28 * 28])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [5]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [14]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 0.76149666 ; accuracy 0.81443334
epoch 1 : loss 0.76038945 ; accuracy 0.8146667
epoch 2 : loss 0.75928736 ; accuracy 0.81488335
epoch 3 : loss 0.7581902 ; accuracy 0.81516665
epoch 4 : loss 0.75709814 ; accuracy 0.81556666
epoch 5 : loss 0.756011 ; accuracy 0.8156833
epoch 6 : loss 0.7549289 ; accuracy 0.8159
epoch 7 : loss 0.75385165 ; accuracy 0.8160667
epoch 8 : loss 0.75277925 ; accuracy 0.81625
epoch 9 : loss 0.75171167 ; accuracy 0.81628335
epoch 10 : loss 0.75064874 ; accuracy 0.8165333
epoch 11 : loss 0.7495905 ; accuracy 0.8167167
epoch 12 : loss 0.74853706 ; accuracy 0.81696665
epoch 13 : loss 0.74748826 ; accuracy 0.81725
epoch 14 : loss 0.74644417 ; accuracy 0.81741667
epoch 15 : loss 0.7454046 ; accuracy 0.81761664
epoch 16 : loss 0.7443697 ; accuracy 0.81771666
epoch 17 : loss 0.7433391 ; accuracy 0.8179833
epoch 18 : loss 0.74231327 ; accuracy 0.81818336
epoch 19 : loss 0.74129206 ; accuracy 0.81843334
epoch 20 : loss 0.7402754 ; accuracy 0.8188
epoch 21 :